# GNN Solver for RAVEN Progressive Matrices using PyNeuraLogic

This notebook implements a **Graph Neural Network** solver for [RAVEN](https://github.com/WellyZhang/RAVEN) / [I-RAVEN](https://github.com/husheng12345/SRAN) progressive matrices using [PyNeuraLogic](https://github.com/LukasZahradnik/PyNeuraLogic) — a framework for **differentiable logic programming** that natively expresses relational (graph) structure.

## Approach

Each RPM puzzle consists of a 3×3 grid of panels. The bottom-right panel is missing, and the task is to select the correct answer from 8 candidates. Each panel contains objects with attributes: **type** (shape), **size**, and **color** at specific **positions**.

We encode each puzzle as a **relational graph**:

- **Nodes**: Each object in each panel becomes a graph node
- **Node features**: Learnable embeddings for object type, size, and color
- **Edges**: Three types of relational edges:
  - `row_edge`: connects objects at the same position across panels in the same row
  - `col_edge`: connects objects at the same position across panels in the same column
  - `intra_edge`: connects objects within the same panel (for multi-object constellations)

The GNN propagates information through these edges, enabling each candidate node to aggregate contextual patterns from the puzzle grid. A scoring head then reads out a scalar per candidate.

This mirrors the existing PyTorch GNN solver in `gnn/` but expresses the architecture as **declarative logic rules** — PyNeuraLogic automatically handles grounding, aggregation, and differentiation.

## 1. Setup and Imports

In [ ]:
import os
import sys
import glob
import time
import random
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

from neuralogic.core import R, V, Template, Settings, Transformation, Aggregation
from neuralogic.dataset import Dataset
from neuralogic.nn import get_evaluator
from neuralogic.optim import Adam
from neuralogic.nn.loss import MSE

print("Imports OK")

## 2. Constants and Data Loading

We use the same constants as the existing GNN solver. The I-RAVEN dataset has 7 constellations, each with panels containing objects at constellation-specific positions mapped to a universal 22-slot position space.

In [ ]:
# ---------- Constants (same as gnn/config.py) ----------
NUM_CONTEXT = 8
NUM_CANDIDATES = 8
NUM_POSITIONS = 22
NUM_SHAPE = 5
NUM_SIZE = 6
NUM_COLOR = 10

CONFIG_POS_MAP = {
    "center_single": {0: 0},
    "distribute_four": {0: 1, 1: 2, 2: 3, 3: 4},
    "distribute_nine": {0: 5, 1: 6, 2: 7, 3: 8, 4: 9, 5: 10, 6: 11, 7: 12, 8: 13},
    "left_center_single_right_center_single": {0: 14, 1: 15},
    "up_center_single_down_center_single": {0: 16, 1: 17},
    "in_center_single_out_center_single": {0: 0, 1: 9},
    "in_distribute_four_out_center_single": {0: 0, 1: 18, 2: 19, 3: 20, 4: 21},
}

ALL_CONFIGS = list(CONFIG_POS_MAP.keys())

CONTEXT_ROWS = [[0, 1, 2], [3, 4, 5]]
CONTEXT_COLS = [[0, 3, 6], [1, 4, 7]]
ANSWER_ROW = [6, 7]   # answer row = [P6, P7, candidate]
ANSWER_COL = [2, 5]   # answer col = [P2, P5, candidate]

DATASET_ROOT = "data/I-RAVEN-10000"

print(f"Dataset root: {DATASET_ROOT}")
print(f"Available configs: {os.listdir(DATASET_ROOT)}")

In [ ]:
# ---------- XML Parsing (from gnn/data.py) ----------

def parse_ground_truth(xml_path, config):
    """Parse RAVEN XML -> per-panel list of objects with universal positions."""
    pos_map = CONFIG_POS_MAP[config]
    tree = ET.parse(xml_path)
    root = tree.getroot()
    xml_panels = root[0]
    gt_panels = []
    for panel in xml_panels:
        panel_objects = []
        global_pos_offset = 0
        for struct in panel:
            for comp in struct:
                for layout in comp:
                    all_positions = eval(layout.attrib["Position"])
                    for entity in layout:
                        bbox = eval(entity.attrib["bbox"])
                        local_pos = all_positions.index(bbox) + global_pos_offset
                        pos = pos_map[local_pos]
                        panel_objects.append({
                            "position": pos,
                            "type": int(entity.attrib["Type"]) - 1,
                            "size": int(entity.attrib["Size"]),
                            "color": int(entity.attrib["Color"]),
                        })
                    global_pos_offset += len(all_positions)
        gt_panels.append(sorted(panel_objects, key=lambda o: o["position"]))
    return gt_panels


def gt_to_dense(gt_panels):
    """Convert parsed GT to dense [16, 22] tensors."""
    obj_type = np.zeros((16, NUM_POSITIONS), dtype=np.int64)
    obj_size = np.zeros((16, NUM_POSITIONS), dtype=np.int64)
    obj_color = np.zeros((16, NUM_POSITIONS), dtype=np.int64)
    obj_exists = np.zeros((16, NUM_POSITIONS), dtype=bool)
    for p, objs in enumerate(gt_panels):
        for o in objs:
            pos = o["position"]
            if 0 <= pos < NUM_POSITIONS:
                obj_type[p, pos] = min(o["type"], NUM_SHAPE - 1)
                obj_size[p, pos] = min(o["size"], NUM_SIZE - 1)
                obj_color[p, pos] = min(o["color"], NUM_COLOR - 1)
                obj_exists[p, pos] = True
    return obj_type, obj_size, obj_color, obj_exists


def load_samples(dataset_root, config, split, max_samples=None):
    """Load RPM samples from I-RAVEN dataset."""
    config_dir = os.path.join(dataset_root, config)
    npz_files = sorted(glob.glob(os.path.join(config_dir, f"*_{split}.npz")))
    if max_samples:
        npz_files = npz_files[:max_samples]

    samples = []
    for npz_path in npz_files:
        data = np.load(npz_path)
        target = int(data["target"])
        xml_path = npz_path.replace(".npz", ".xml")
        if not os.path.isfile(xml_path):
            continue
        gt_panels = parse_ground_truth(xml_path, config)
        obj_type, obj_size, obj_color, obj_exists = gt_to_dense(gt_panels)
        samples.append((obj_type, obj_size, obj_color, obj_exists, target))
    return samples

print("Data loading functions defined.")

In [ ]:
# ---------- Load and inspect a sample ----------
config = "center_single"
sample = load_samples(DATASET_ROOT, config, "train", max_samples=1)[0]
obj_type, obj_size, obj_color, obj_exists, target = sample

print(f"Config: {config}")
print(f"Target (correct candidate): {target}")
print(f"Object exists shape: {obj_exists.shape}")
print(f"\nObjects per panel:")
for p in range(16):
    positions = np.where(obj_exists[p])[0]
    role = "context" if p < 8 else f"candidate {p - 8}"
    if len(positions) > 0:
        attrs = [(int(obj_type[p, pos]), int(obj_size[p, pos]), int(obj_color[p, pos])) for pos in positions]
        print(f"  Panel {p:2d} ({role:14s}): pos={list(positions)}, (type,size,color)={attrs}")

## 3. Graph Encoding: RPM → PyNeuraLogic Facts

Each RPM puzzle is encoded as a relational graph with the following structure:

### Node identity
Each object is identified by `"p{panel}_o{position}"`. For `center_single`, every panel has one object at position 0, so nodes are `"p0_o0"` through `"p15_o0"`.

### Node attribute facts
For each object node, we assert its type, size, and color:
- `R.t{val}(node_id)` — shape type (0–4)
- `R.s{val}(node_id)` — size (0–5)
- `R.c{val}(node_id)` — color (0–9)

### Edge facts
Three types of directed edges capture the RPM relational structure:

1. **`row_edge(X, Y)`** — connects objects at the same position in panels that share a row in the 3×3 grid. This enables the GNN to detect row-wise patterns (constant, progression, etc.).

2. **`col_edge(X, Y)`** — same as above but for columns.

3. **`intra_edge(X, Y)`** — connects objects within the same panel (for multi-object constellations like `distribute_four`).

### Candidate marking
- `R.is_cand(node_id, candidate_index)` — marks which candidate (0–7) each answer-panel object belongs to. Used by the scoring head.

### Edge connectivity for candidates
Each candidate panel (8+c) occupies the bottom-right cell of the 3×3 grid. It connects to:
- **Row 2**: context panels 6 and 7 (via `row_edge`)
- **Column 2**: context panels 2 and 5 (via `col_edge`)

Candidates do **not** connect to each other — they are independent alternatives.

In [ ]:
def rpm_sample_to_facts(obj_type, obj_size, obj_color, obj_exists):
    """
    Convert one RPM sample's [16, 22] attribute tensors into PyNeuraLogic facts.

    Returns a list of facts encoding:
    - Object attribute facts (type, size, color per node)
    - Row/column/intra-panel edges
    - Candidate markers for the scoring head
    """
    facts = []

    def nid(panel, pos):
        return f"p{panel}_o{pos}"

    # --- Object attribute facts ---
    for panel in range(16):
        for pos in range(NUM_POSITIONS):
            if not obj_exists[panel, pos]:
                continue
            n = nid(panel, pos)
            facts.append(R.get(f"t{int(obj_type[panel, pos])}")(n))
            facts.append(R.get(f"s{int(obj_size[panel, pos])}")(n))
            facts.append(R.get(f"c{int(obj_color[panel, pos])}")(n))

    # --- Helper: add bidirectional edges between objects at shared positions ---
    def add_cross_panel_edges(panel_i, panel_j, rel_name):
        rel = R.get(rel_name)
        for pos in range(NUM_POSITIONS):
            if obj_exists[panel_i, pos] and obj_exists[panel_j, pos]:
                facts.append(rel(nid(panel_i, pos), nid(panel_j, pos)))
                facts.append(rel(nid(panel_j, pos), nid(panel_i, pos)))

    # --- Context row edges (rows 0 and 1) ---
    for row in CONTEXT_ROWS:  # [0,1,2] and [3,4,5]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                add_cross_panel_edges(row[i], row[j], "row_edge")

    # --- Context column edges (cols 0 and 1) ---
    for col in CONTEXT_COLS:  # [0,3,6] and [1,4,7]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                add_cross_panel_edges(col[i], col[j], "col_edge")

    # --- Partial row 2 and col 2 among context panels ---
    add_cross_panel_edges(6, 7, "row_edge")  # P6 <-> P7
    add_cross_panel_edges(2, 5, "col_edge")  # P2 <-> P5

    # --- Candidate edges: each candidate connects to its row/col context ---
    for ci in range(NUM_CANDIDATES):
        cand_panel = NUM_CONTEXT + ci
        # Row 2: candidate <-> P6, candidate <-> P7
        add_cross_panel_edges(cand_panel, 6, "row_edge")
        add_cross_panel_edges(cand_panel, 7, "row_edge")
        # Col 2: candidate <-> P2, candidate <-> P5
        add_cross_panel_edges(cand_panel, 2, "col_edge")
        add_cross_panel_edges(cand_panel, 5, "col_edge")

    # --- Intra-panel edges (all objects within same panel, for multi-object configs) ---
    for panel in range(16):
        positions = [pos for pos in range(NUM_POSITIONS) if obj_exists[panel, pos]]
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                facts.append(R.intra_edge(nid(panel, positions[i]), nid(panel, positions[j])))
                facts.append(R.intra_edge(nid(panel, positions[j]), nid(panel, positions[i])))

    # --- Candidate markers ---
    for ci in range(NUM_CANDIDATES):
        cand_panel = NUM_CONTEXT + ci
        for pos in range(NUM_POSITIONS):
            if obj_exists[cand_panel, pos]:
                facts.append(R.is_cand(nid(cand_panel, pos), ci))

    return facts

print("Fact construction function defined.")

In [ ]:
# Demonstrate facts for one sample
demo_facts = rpm_sample_to_facts(obj_type, obj_size, obj_color, obj_exists)

attr_facts = [f for f in demo_facts if "edge" not in str(f) and "cand" not in str(f)]
edge_facts = [f for f in demo_facts if "edge" in str(f)]
cand_facts = [f for f in demo_facts if "cand" in str(f)]

print(f"Total facts: {len(demo_facts)}")
print(f"  Attribute facts: {len(attr_facts)}")
print(f"  Edge facts:      {len(edge_facts)}")
print(f"  Candidate marks: {len(cand_facts)}")
print(f"\nSample attribute facts (first 6):")
for f in attr_facts[:6]:
    print(f"  {f}")
print(f"\nSample edge facts (first 6):")
for f in edge_facts[:6]:
    print(f"  {f}")
print(f"\nCandidate markers:")
for f in cand_facts:
    print(f"  {f}")

## 4. GNN Template Definition

The GNN is expressed as a set of **weighted logic rules** in PyNeuraLogic. This follows the same pattern as the Mutagenesis GNN example but adapted for RPM:

1. **Attribute embeddings**: Each discrete attribute value (type 0–4, size 0–5, color 0–9) maps to a learnable embedding vector of dimension `D`.

2. **Node embedding `h0`**: Sum of the three attribute embeddings for each object.

3. **GNN Layer 1 `h1`**: Message passing along `row_edge`, `col_edge`, and `intra_edge`. Each edge type has its own learnable weight matrices. A self-connection ensures all nodes get updated.

4. **GNN Layer 2 `h2`**: Same structure, deeper propagation. After 2 hops, each candidate node has aggregated information from all context panels.

5. **Scoring head `score(C)`**: For each candidate index `C`, pool the GNN outputs of that candidate's objects through a learnable projection to produce a scalar score.

In [ ]:
def build_rpm_gnn_template(embed_dim=8):
    """Build a PyNeuraLogic template encoding a 2-layer relational GNN for RPM solving."""
    D = embed_dim
    template = Template()

    # --- Attribute embeddings ---
    # Each attribute value gets a unique learnable vector of dimension D.
    # This is analogous to atom_embed in the Mutagenesis GNN.
    template.add_rules([
        R.type_emb(V.X)[D,] <= R.get(f"t{t}")(V.X) for t in range(NUM_SHAPE)
    ])
    template.add_rules([
        R.size_emb(V.X)[D,] <= R.get(f"s{s}")(V.X) for s in range(NUM_SIZE)
    ])
    template.add_rules([
        R.color_emb(V.X)[D,] <= R.get(f"c{c}")(V.X) for c in range(NUM_COLOR)
    ])

    # --- Initial node embedding: combine attribute embeddings ---
    template += R.h0(V.X) <= (R.type_emb(V.X), R.size_emb(V.X), R.color_emb(V.X))

    # --- GNN Layer 1: multi-relational message passing ---
    # Row edges: propagate along same-row panels
    template += R.h1(V.X) <= (R.h0(V.X)[D, D], R.h0(V.Y)[D, D], R.row_edge(V.X, V.Y))
    # Column edges: propagate along same-column panels
    template += R.h1(V.X) <= (R.h0(V.X)[D, D], R.h0(V.Y)[D, D], R.col_edge(V.X, V.Y))
    # Intra-panel edges: propagate within same panel
    template += R.h1(V.X) <= (R.h0(V.X)[D, D], R.h0(V.Y)[D, D], R.intra_edge(V.X, V.Y))
    # Self-connection: ensures h1 is defined even for isolated nodes
    template += R.h1(V.X) <= R.h0(V.X)[D, D]

    # --- GNN Layer 2: deeper propagation ---
    template += R.h2(V.X) <= (R.h1(V.X)[D, D], R.h1(V.Y)[D, D], R.row_edge(V.X, V.Y))
    template += R.h2(V.X) <= (R.h1(V.X)[D, D], R.h1(V.Y)[D, D], R.col_edge(V.X, V.Y))
    template += R.h2(V.X) <= (R.h1(V.X)[D, D], R.h1(V.Y)[D, D], R.intra_edge(V.X, V.Y))
    template += R.h2(V.X) <= R.h1(V.X)[D, D]

    # --- Scoring head: project each candidate's pooled GNN output to a scalar ---
    # For candidate C, aggregate h2 of all objects marked as belonging to candidate C,
    # then project through a [1, D] learnable weight to produce score(C).
    template += R.score(V.C)[1, D] <= (R.h2(V.X), R.is_cand(V.X, V.C))

    return template


EMBED_DIM = 8
template = build_rpm_gnn_template(EMBED_DIM)
print(template)

## 5. Dataset Construction

For each RPM sample, we create:
- **One example** (set of facts) describing all 16 panels and their graph connections
- **Eight queries** `score(0)` through `score(7)`, labeled `1.0` for the correct candidate and `0.0` for the rest

PyNeuraLogic's evaluator treats each query as a separate training signal (MSE loss against the label). The model learns to assign high scores to correct candidates.

In [ ]:
def build_pynl_dataset(samples):
    """Convert a list of RPM samples into a PyNeuraLogic Dataset."""
    dataset = Dataset()
    for obj_type, obj_size, obj_color, obj_exists, target in samples:
        facts = rpm_sample_to_facts(obj_type, obj_size, obj_color, obj_exists)
        dataset.add_example(facts)
        queries = []
        for ci in range(NUM_CANDIDATES):
            label = 1.0 if ci == target else 0.0
            queries.append(R.score(ci)[label])
        dataset.add_queries(queries)
    return dataset

print("Dataset builder defined.")

In [ ]:
# ---------- Load data for one constellation ----------
CONFIG = "center_single"
N_TRAIN = 500
N_VAL = 200
N_TEST = 200

print(f"Loading {CONFIG} samples...")
t0 = time.time()
train_samples = load_samples(DATASET_ROOT, CONFIG, "train", max_samples=N_TRAIN)
val_samples = load_samples(DATASET_ROOT, CONFIG, "val", max_samples=N_VAL)
test_samples = load_samples(DATASET_ROOT, CONFIG, "test", max_samples=N_TEST)
print(f"Loaded: {len(train_samples)} train, {len(val_samples)} val, {len(test_samples)} test  [{time.time()-t0:.1f}s]")

print("\nBuilding PyNeuraLogic datasets...")
t0 = time.time()
train_dataset = build_pynl_dataset(train_samples)
val_dataset = build_pynl_dataset(val_samples)
test_dataset = build_pynl_dataset(test_samples)
print(f"Built: train={len(train_samples)*8} queries, val={len(val_samples)*8}, test={len(test_samples)*8}  [{time.time()-t0:.1f}s]")

## 6. Training

We use PyNeuraLogic's built-in evaluator with MSE loss and Adam optimizer. The evaluator:
1. **Grounds** the template rules against each example's facts (creates sample-specific neural networks)
2. **Trains** by backpropagating through the grounded networks

Grounding may take a few minutes for the first build, but this is a one-time cost.

In [ ]:
NUM_EPOCHS = 50
LR = 0.001

settings = Settings(optimizer=Adam(lr=LR), epochs=NUM_EPOCHS, error_function=MSE())
evaluator = get_evaluator(template, settings)

print("Building (grounding) training dataset... this may take a few minutes.")
t0 = time.time()
built_train = evaluator.build_dataset(train_dataset)
print(f"Grounding complete: {len(built_train.samples)} grounded samples [{time.time()-t0:.1f}s]")

print("\nBuilding validation dataset...")
t0 = time.time()
built_val = evaluator.build_dataset(val_dataset)
print(f"Grounding complete: {len(built_val.samples)} grounded samples [{time.time()-t0:.1f}s]")

In [ ]:
def evaluate_accuracy(evaluator, built_dataset, num_rpm_samples):
    """Evaluate RPM accuracy: for each puzzle, pick the candidate with the highest predicted score."""
    preds = list(evaluator.test(built_dataset))
    correct = 0
    for i in range(num_rpm_samples):
        scores = preds[i * 8 : (i + 1) * 8]
        predicted = int(np.argmax(scores))
        target_label = built_dataset.samples[i * 8 + 0].target
        # Find the correct candidate (the one with label 1.0)
        for ci in range(8):
            if built_dataset.samples[i * 8 + ci].target > 0.5:
                actual = ci
                break
        if predicted == actual:
            correct += 1
    return correct / num_rpm_samples if num_rpm_samples > 0 else 0.0


print("Training...")
train_losses = []
val_accs = []
best_val_acc = 0.0

for epoch_loss, n_samples in evaluator.train(built_train):
    avg_loss = epoch_loss / n_samples
    train_losses.append(avg_loss)
    epoch = len(train_losses)

    # Evaluate every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        val_acc = evaluate_accuracy(evaluator, built_val, len(val_samples))
        val_accs.append((epoch, val_acc))
        best_val_acc = max(best_val_acc, val_acc)
        clear_output(wait=True)
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc*100:.1f}%  best={best_val_acc*100:.1f}%")

print(f"\nTraining complete. Best validation accuracy: {best_val_acc*100:.1f}%")

# conda run -n nvsa python -c "
# from neuralogic.nn.loss import MSE, CrossEntropy
# print('CrossEntropy available')
# import neuralogic.nn.loss as L
# print([x for x in dir(L) if not x.startswith('_')])
# " 2>&1

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")

if val_accs:
    epochs_v, accs_v = zip(*val_accs)
    ax2.plot(epochs_v, [a * 100 for a in accs_v], 'o-')
    ax2.axhline(y=12.5, color='r', linestyle='--', label='Random (12.5%)')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title(f"Validation Accuracy ({CONFIG})")
    ax2.legend()

plt.tight_layout()
plt.show()

## 7. Test Evaluation

In [ ]:
print("Building test dataset...")
t0 = time.time()
built_test = evaluator.build_dataset(test_dataset)
print(f"Grounding complete [{time.time()-t0:.1f}s]")

test_acc = evaluate_accuracy(evaluator, built_test, len(test_samples))
print(f"\nTest accuracy on {CONFIG}: {test_acc*100:.1f}%  ({int(test_acc * len(test_samples))}/{len(test_samples)})")
print(f"Random baseline: 12.5%")

## 8. Multi-Constellation Evaluation

The graph encoding generalizes to all 7 I-RAVEN constellations. Multi-object constellations (e.g., `distribute_four` with 4 objects per panel) produce richer graphs with intra-panel edges.

Below we evaluate on multiple constellations using a model trained per-constellation.

In [ ]:
def train_and_evaluate_config(
    config,
    dataset_root=DATASET_ROOT,
    n_train=200,
    n_test=100,
    embed_dim=8,
    epochs=30,
    lr=0.001,
):
    """Train and evaluate a PyNeuraLogic GNN on a single constellation."""
    print(f"\n{'='*60}")
    print(f"  Config: {config}")
    print(f"{'='*60}")

    train_samples = load_samples(dataset_root, config, "train", max_samples=n_train)
    test_samples = load_samples(dataset_root, config, "test", max_samples=n_test)
    if not train_samples or not test_samples:
        print(f"  Skipping — no data found.")
        return None
    print(f"  Loaded: {len(train_samples)} train, {len(test_samples)} test")

    train_ds = build_pynl_dataset(train_samples)
    test_ds = build_pynl_dataset(test_samples)

    tmpl = build_rpm_gnn_template(embed_dim)
    sett = Settings(optimizer=Adam(lr=lr), epochs=epochs, error_function=MSE())
    ev = get_evaluator(tmpl, sett)

    print(f"  Grounding train...")
    t0 = time.time()
    built_tr = ev.build_dataset(train_ds)
    print(f"  Grounded {len(built_tr.samples)} samples [{time.time()-t0:.1f}s]")

    print(f"  Training {epochs} epochs...")
    t0 = time.time()
    for loss, n in ev.train(built_tr):
        pass
    print(f"  Training done [{time.time()-t0:.1f}s]")

    print(f"  Grounding test...")
    t0 = time.time()
    built_te = ev.build_dataset(test_ds)
    print(f"  Grounded {len(built_te.samples)} samples [{time.time()-t0:.1f}s]")

    acc = evaluate_accuracy(ev, built_te, len(test_samples))
    print(f"  Test accuracy: {acc*100:.1f}%  ({int(acc*len(test_samples))}/{len(test_samples)})")
    return acc

In [ ]:
# Evaluate on selected constellations
# (reduce n_train/n_test for faster experimentation; increase for better results)
configs_to_eval = [
    "center_single",
    "up_center_single_down_center_single",
    "left_center_single_right_center_single",
    "in_center_single_out_center_single",
    "distribute_four",
]

results = {}
for cfg in configs_to_eval:
    acc = train_and_evaluate_config(
        cfg,
        n_train=200,
        n_test=100,
        embed_dim=8,
        epochs=30,
        lr=0.001,
    )
    if acc is not None:
        results[cfg] = acc

print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for cfg, acc in results.items():
    print(f"  {cfg:50s}  {acc*100:6.1f}%")
if results:
    overall = np.mean(list(results.values()))
    print(f"  {'MEAN':50s}  {overall*100:6.1f}%")

## 9. Comparison with Existing Solvers

| Solver | Approach | Trainable | Uses GT? |
|--------|----------|-----------|----------|
| `neuralogic_solver/solver.py` `HardLogicSolver` | Explicit ternary rule checking (hard boolean) | No | Yes |
| `neuralogic_solver/solver.py` `SoftLogicSolver` | Differentiable rule satisfaction probabilities | Yes (rule/attr weights) | Yes |
| `neuralogic_solver/pynl_solver.py` `PynlHardSolver` | PyNeuraLogic constraint satisfaction (inference engine) | No | Yes |
| `gnn/model.py` `CandidateVsaGNN` | PyTorch GNN with relational edges + attention | Yes (full GNN) | Yes / Images |
| **This notebook** | PyNeuraLogic declarative GNN with relational edges | Yes (GNN weights) | Yes |

### Key differences from the PyTorch GNN (`gnn/`):

1. **Declarative**: The GNN architecture is specified as logic rules rather than imperative PyTorch modules. PyNeuraLogic handles grounding and backpropagation.

2. **Variable-structure**: Each sample gets its own grounded network topology based on which facts are present. No padding or masking needed.

3. **Relational inductive bias**: Edge types (row, column, intra-panel) are first-class relations in the logic program, each with separate learnable weights.

### To improve performance:
- Increase `embed_dim` (e.g., 16, 32)
- Add more GNN layers (h3, h4)
- Train on more samples and more epochs
- Add a context pooling term to the score head
- Use cross-entropy loss instead of MSE (requires custom training loop)